**Table of contents**<a id='toc0_'></a>    
- [1 利用 `loc` 函数和布尔列表进行筛选](#toc1_)    
- [2 利用 `merge` 函数筛选](#toc2_)    
- [3 利用 `query` 函数进行筛选](#toc3_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

在利用 Pandas 进行数据分析的过程当中，我们常常会碰到需要实现复杂查询或者筛选的场景。比如说从全样本股票中，剔除 ST 和 \*ST 股票，然后筛选属于中小板的股票。这篇文章总结我在科研实践过程当中，使用 Pandas 实现复杂筛选的方法。

<!-- TEASER_END -->

首先导入 Pandas 和 tushare，以下的演示数据来自于从 tushare 调用的上海证券交易所的上市公司数据。

In [1]:
import numpy as np
import pandas as pd
import tushare as ts

pro = ts.pro_api()

In [2]:
data = pro.stock_basic(exchange='SSE', list_status='L', fields='ts_code,symbol,name,area,industry,list_date')

In [3]:
data.shape

(2197, 6)

In [4]:
data.head()

,ts_code,symbol,name,area,industry,list_date
0,600000.SH,600000,浦发银行,上海,银行,19991110
1,600004.SH,600004,白云机场,广东,机场,20030428
2,600006.SH,600006,东风汽车,湖北,汽车整车,19990727
3,600007.SH,600007,中国国贸,北京,园区开发,19990312
4,600008.SH,600008,首创环保,北京,环境保护,20000427


# <a id='toc1_'></a>[1 利用 `loc` 函数和布尔列表进行筛选](#toc0_)

我最常用的一个做法是使用 `loc` 函数和布尔列表进行筛选。`loc`函数可以接受布尔列表作为参数，而根据实际需求，我们有多种方式可以生成布尔列表，最普遍的是利用逻辑运算来生成布尔列表。

- 逻辑运算产生布尔列表

比如，我们需要筛选出注册地在北京的股票，使用如下逻辑运算，可以产生一个布尔列表。

In [5]:
data.area == "北京"

0       False
1       False
2       False
3        True
4        True
        ...  
2192    False
2193    False
2194    False
2195    False
2196     True
Name: area, Length: 2197, dtype: bool

然后我们将这个布尔列表传入 `loc` 函数中，就可以筛选出注册地在北京的股票。

In [6]:
data.loc[data.area == "北京", :]

,ts_code,symbol,name,area,industry,list_date
3,600007.SH,600007,中国国贸,北京,园区开发,19990312
4,600008.SH,600008,首创环保,北京,环境保护,20000427
7,600011.SH,600011,华能国际,北京,火力发电,20011206
9,600015.SH,600015,华夏银行,北京,银行,20030912
10,600016.SH,600016,民生银行,北京,银行,20001219
...,...,...,...,...,...,...
2140,688658.SH,688658,悦康药业,北京,化学制药,20201224
2161,688687.SH,688687,凯因科技,北京,生物制药,20210208
2173,688722.SH,688722,同益中,北京,化纤,20211019
2187,688787.SH,688787,海天瑞声,北京,软件服务,20210813


也可以使用复杂的逻辑运算，比如我们筛选注册地在北京的银行股票。

In [7]:
data.loc[(data.area=="北京") & (data.industry=="银行"), :]

,ts_code,symbol,name,area,industry,list_date
9,600015.SH,600015,华夏银行,北京,银行,20030912
10,600016.SH,600016,民生银行,北京,银行,20001219
846,601169.SH,601169,北京银行,北京,银行,20070919
871,601288.SH,601288,农业银行,北京,银行,20100715
890,601398.SH,601398,工商银行,北京,银行,20061027
923,601658.SH,601658,邮储银行,北京,银行,20191210
954,601818.SH,601818,光大银行,北京,银行,20100818
987,601939.SH,601939,建设银行,北京,银行,20070925
999,601988.SH,601988,中国银行,北京,银行,20060705
1007,601998.SH,601998,中信银行,北京,银行,20070427


使用 `loc` 函数也很容易实现对列的筛选，我们只需要将需要的列以列表形式传入即可。

In [8]:
data.loc[(data.area=="北京")&(data.industry=="银行"), ["symbol", "name", "list_date"]]

,symbol,name,list_date
9,600015,华夏银行,20030912
10,600016,民生银行,20001219
846,601169,北京银行,20070919
871,601288,农业银行,20100715
890,601398,工商银行,20061027
923,601658,邮储银行,20191210
954,601818,光大银行,20100818
987,601939,建设银行,20070925
999,601988,中国银行,20060705
1007,601998,中信银行,20070427


- 函数判断产生布尔变量

第二种常用的手段是使用函数判断生成布尔变量。比如我们挑选上市日期在 2000 年的股票，这里使用 `startswith` 函数来对上市日期这一列做判断。

In [9]:
data.loc[[date.startswith("2000") for date in data.list_date]]

,ts_code,symbol,name,area,industry,list_date
4,600008.SH,600008,首创环保,北京,环境保护,20000427
10,600016.SH,600016,民生银行,北京,银行,20001219
13,600019.SH,600019,宝钢股份,上海,普钢,20001212
30,600038.SH,600038,中直股份,黑龙江,航空,20001218
101,600130.SH,600130,波导股份,浙江,通信设备,20000706
...,...,...,...,...,...,...
308,600388.SH,600388,ST龙净,福建,环境保护,20001229
317,600398.SH,600398,海澜之家,江苏,服饰,20001228
318,600399.SH,600399,抚顺特钢,辽宁,特种钢,20001229
332,600422.SH,600422,昆药集团,云南,中成药,20001206


再比如，剔除 ST 股。

In [10]:
data.loc[~np.array([name.startswith("ST") for name in data.name]), :]

,ts_code,symbol,name,area,industry,list_date
0,600000.SH,600000,浦发银行,上海,银行,19991110
1,600004.SH,600004,白云机场,广东,机场,20030428
2,600006.SH,600006,东风汽车,湖北,汽车整车,19990727
3,600007.SH,600007,中国国贸,北京,园区开发,19990312
4,600008.SH,600008,首创环保,北京,环境保护,20000427
...,...,...,...,...,...,...
2192,688799.SH,688799,华纳药厂,湖南,化学制药,20210713
2193,688800.SH,688800,瑞可达,江苏,元器件,20210722
2194,688819.SH,688819,天能股份,浙江,电气设备,20210118
2195,688981.SH,688981,中芯国际,上海,半导体,20200716


我们也可以自定义函数，来实现更复杂的判断需求。比如，对于上面的剔除 ST 股，我们也可以通过自定义函数实现：

In [11]:
def filter_ST(name):
    return not name.startswith("ST")

In [12]:
data.loc[data.loc[:,'name'].apply(lambda name: filter_ST(name)), :]

,ts_code,symbol,name,area,industry,list_date
0,600000.SH,600000,浦发银行,上海,银行,19991110
1,600004.SH,600004,白云机场,广东,机场,20030428
2,600006.SH,600006,东风汽车,湖北,汽车整车,19990727
3,600007.SH,600007,中国国贸,北京,园区开发,19990312
4,600008.SH,600008,首创环保,北京,环境保护,20000427
...,...,...,...,...,...,...
2192,688799.SH,688799,华纳药厂,湖南,化学制药,20210713
2193,688800.SH,688800,瑞可达,江苏,元器件,20210722
2194,688819.SH,688819,天能股份,浙江,电气设备,20210118
2195,688981.SH,688981,中芯国际,上海,半导体,20200716


- `eval` 函数产生布尔变量

我们也可以使用 `eval` 函数来产生布尔变量。比如，筛选注册地在北京的银行：

In [13]:
data.loc[data.eval("area=='北京' and industry=='银行'"), :]

,ts_code,symbol,name,area,industry,list_date
9,600015.SH,600015,华夏银行,北京,银行,20030912
10,600016.SH,600016,民生银行,北京,银行,20001219
846,601169.SH,601169,北京银行,北京,银行,20070919
871,601288.SH,601288,农业银行,北京,银行,20100715
890,601398.SH,601398,工商银行,北京,银行,20061027
923,601658.SH,601658,邮储银行,北京,银行,20191210
954,601818.SH,601818,光大银行,北京,银行,20100818
987,601939.SH,601939,建设银行,北京,银行,20070925
999,601988.SH,601988,中国银行,北京,银行,20060705
1007,601998.SH,601998,中信银行,北京,银行,20070427


# <a id='toc2_'></a>[2 利用 `merge` 函数筛选](#toc0_)

有时候，我们需要根据样本来筛选数据，这时候可以使用 `merge` 函数来进行合并，合并过程也就是数据的筛选过程。比如，我们需要对样本股票的历史数据进行回归，需要从全样本历史数据中筛选出样本股票的历史收益率数据。下面的例子，我们首选读取全样本股票的历史收益率数据，然后筛选出注册地在北京的银行股票的历史收益率数据。

In [14]:
r = pd.read_csv("TRD_Dalyr2.csv", dtype={'Stkcd':str})

In [15]:
r.head()

,Stkcd,Trddt,Dretwd
0,300587,2018-09-28,0.046127
1,300587,2018-10-08,-0.042397
2,300587,2018-10-09,-0.004132
3,300587,2018-10-10,0.011855
4,300587,2018-10-11,0.100176


In [16]:
banks = data.loc[(data.area=="北京")&(data.industry=="银行"), ["symbol", "name", "list_date"]]

In [17]:
banks

,symbol,name,list_date
9,600015,华夏银行,20030912
10,600016,民生银行,20001219
846,601169,北京银行,20070919
871,601288,农业银行,20100715
890,601398,工商银行,20061027
923,601658,邮储银行,20191210
954,601818,光大银行,20100818
987,601939,建设银行,20070925
999,601988,中国银行,20060705
1007,601998,中信银行,20070427


In [18]:
r.merge(banks.symbol, left_on="Stkcd", right_on="symbol", how="inner")

,Stkcd,Trddt,Dretwd,symbol
0,600015,2018-01-02,0.012222,600015
1,600015,2018-01-03,0.002195,600015
2,600015,2018-01-04,0.002191,600015
3,600015,2018-01-05,0.001093,600015
4,600015,2018-01-08,0.003275,600015
...,...,...,...,...
2035,600016,2022-03-10,-0.002653,600016
2036,600016,2022-03-11,0.002660,600016
2037,600016,2022-03-14,-0.007958,600016
2038,600016,2022-03-15,-0.045455,600016


# <a id='toc3_'></a>[3 利用 `query` 函数进行筛选](#toc0_)

`query` 是 Pandas 提供的一个方法，使用布尔表达式对 DataFrame 进行查询，这个布尔表达式支持一个文本字符串。比如，查询注册地在北京的银行：

In [19]:
data.query("area=='北京' and industry == '银行'")

,ts_code,symbol,name,area,industry,list_date
9,600015.SH,600015,华夏银行,北京,银行,20030912
10,600016.SH,600016,民生银行,北京,银行,20001219
846,601169.SH,601169,北京银行,北京,银行,20070919
871,601288.SH,601288,农业银行,北京,银行,20100715
890,601398.SH,601398,工商银行,北京,银行,20061027
923,601658.SH,601658,邮储银行,北京,银行,20191210
954,601818.SH,601818,光大银行,北京,银行,20100818
987,601939.SH,601939,建设银行,北京,银行,20070925
999,601988.SH,601988,中国银行,北京,银行,20060705
1007,601998.SH,601998,中信银行,北京,银行,20070427


`query` 也支持传入变量，使用 `@` 传入。

In [20]:
city = "北京"
data.query("area == @city")

,ts_code,symbol,name,area,industry,list_date
3,600007.SH,600007,中国国贸,北京,园区开发,19990312
4,600008.SH,600008,首创环保,北京,环境保护,20000427
7,600011.SH,600011,华能国际,北京,火力发电,20011206
9,600015.SH,600015,华夏银行,北京,银行,20030912
10,600016.SH,600016,民生银行,北京,银行,20001219
...,...,...,...,...,...,...
2140,688658.SH,688658,悦康药业,北京,化学制药,20201224
2161,688687.SH,688687,凯因科技,北京,生物制药,20210208
2173,688722.SH,688722,同益中,北京,化纤,20211019
2187,688787.SH,688787,海天瑞声,北京,软件服务,20210813
